# Trading Strategy Development Notebook

This notebook is for developing and testing trading strategies before deployment.

In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf
from datetime import datetime, timedelta
import matplotlib.pyplot as plt

# Fetch data
symbol = 'EUR/USD'
start = '2021-01-01'
end = datetime.now().strftime('%Y-%m-%d')

df = yf.download('EURUSD=X', start=start, end=end, interval='1d')
print(f'Downloaded {len(df)} candles for {symbol}')

In [ ]:
# Calculate RSI
def calculate_rsi(prices, period=14):
    delta = prices.diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=period).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=period).mean()
    rs = gain / loss
    rsi = 100 - (100 / (1 + rs))
    return rsi

df['RSI'] = calculate_rsi(df['Close'])
print(df[['Close', 'RSI']].tail())

In [ ]:
# Simple mean reversion strategy
df['Signal'] = 0
df.loc[df['RSI'] < 30, 'Signal'] = 1  # Buy signal
df.loc[df['RSI'] > 70, 'Signal'] = -1  # Sell signal

# Calculate returns
df['Returns'] = df['Close'].pct_change()
df['Strategy_Returns'] = df['Signal'].shift(1) * df['Returns']
df['Cumulative_Returns'] = (1 + df['Strategy_Returns']).cumprod()

# Plot
plt.figure(figsize=(12, 6))
plt.plot(df.index, df['Cumulative_Returns'], label='Strategy')
plt.plot(df.index, (1 + df['Returns']).cumprod(), label='Buy & Hold')
plt.legend()
plt.title('Strategy Performance')
plt.show()